# Rectified Flow 动漫头像 V1.2：Google Colab 训练

本 Notebook 用于从 Google Drive 中的项目 ZIP 启动 V1.2。项目代码和数据会解压到 Colab 的 `/content` 临时磁盘，以提高大量小图片的读取速度；数据清单、pHash 去重报告、checkpoint、生成图片、loss 图、TensorBoard 日志和 FID/KID 指标都会持久化到 Google Drive：

```text
MyDrive/KRM_RF_Anime_Colab_Results_V1.2
```

> 请先在 Colab 菜单中选择“运行时 → 更改运行时类型 → T4 GPU”。V1.2 的 Scale-Shift 结构与旧 checkpoint 不兼容，不要把 V1.0/V1.1 权重放入这个结果目录。

## 1. 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 设置 ZIP、结果目录和训练参数

通常只需要确认 `DRIVE_ZIP_PATH` 与上传到 Drive 的实际文件名一致。结果目录固定为 `KRM_RF_Anime_Colab_Results_V1.2`。`TARGET_EPOCHS` 表示最终目标 epoch，不是本次额外训练的轮数；断线重连后保持相同结果目录即可从 `latest.pt` 自动恢复。

In [ ]:
from pathlib import Path

DRIVE_ZIP_PATH = Path('/content/drive/MyDrive/KRM_RF_Anime Images.zip')
DRIVE_RESULTS_DIR = Path('/content/drive/MyDrive/KRM_RF_Anime_Colab_Results_V1.2')

TARGET_EPOCHS = 100
BATCH_SIZE = 32
FORCE_REBUILD_SPLITS = False  # 调整 pHash 参数后才改为 True

assert DRIVE_ZIP_PATH.is_file(), f'找不到项目 ZIP：{DRIVE_ZIP_PATH}'
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('项目 ZIP：', DRIVE_ZIP_PATH)
print('结果目录：', DRIVE_RESULTS_DIR)

## 3. 解压项目到 Colab 临时磁盘

每次获得新的 Colab 运行时都需要重新解压。这里只会清理 `/content/krm_rf_anime_v1_2_workspace`，不会修改 Drive 中的 ZIP 或结果目录。

In [ ]:
import shutil
import zipfile

EXTRACT_ROOT = Path('/content/krm_rf_anime_v1_2_workspace')
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as archive:
    archive.extractall(EXTRACT_ROOT)

project_candidates = sorted({
    train_file.parent
    for train_file in EXTRACT_ROOT.rglob('train.py')
    if (train_file.parent / 'config' / 'default.yaml').is_file()
    and (train_file.parent / 'requirements.txt').is_file()
}, key=lambda path: (len(path.parts), str(path)))
if not project_candidates:
    raise FileNotFoundError('ZIP 中没有找到同时包含 train.py、config/default.yaml 和 requirements.txt 的项目目录')

PROJECT_DIR = project_candidates[0]
DATA_DIR = PROJECT_DIR / 'Data' / 'train' / 'nolabel'
assert DATA_DIR.is_dir(), f'找不到训练图片目录：{DATA_DIR}'
print('项目目录：', PROJECT_DIR)
print('数据目录：', DATA_DIR)
print('PNG 数量：', len(list(DATA_DIR.glob('*.png'))))

## 4. 检查 GPU 和 Colab 环境

In [ ]:
import platform
import subprocess
import sys
import torch

print('Python：', sys.version.split()[0])
print('系统：', platform.platform())
print('PyTorch：', torch.__version__)
print('CUDA 可用：', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('当前没有 GPU，请切换到 T4 GPU 运行时后重新连接')
print('GPU：', torch.cuda.get_device_name(0))
print('GPU 显存：', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GiB')
subprocess.run(['nvidia-smi'], check=True)

## 5. 安装 V1.2 依赖

保留 Colab 自带的 CUDA 版 PyTorch/torchvision，只安装项目工具包以及 V1.2 使用的 ImageHash、torchmetrics 和 torch-fidelity。

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements.txt')],
    check=True,
)

import imagehash
import torch_fidelity
import torchmetrics
import torchvision
import yaml

print('torch：', torch.__version__)
print('torchvision：', torchvision.__version__)
print('torchmetrics：', torchmetrics.__version__)
print('ImageHash：', imagehash.__version__)

## 6. 生成 Drive 专用 V1.2 配置

原始图片继续从 Colab 临时磁盘读取。所有需要持久化的内容都写入 Drive 结果目录，不改写 ZIP 内的默认配置。

In [ ]:
with (PROJECT_DIR / 'config' / 'default.yaml').open('r', encoding='utf-8') as file:
    config = yaml.safe_load(file)

config['project']['name'] = 'rectified-flow-anime-v1.2-colab'
config['data']['raw_dir'] = str(DATA_DIR)
config['data']['split_dir'] = str(DRIVE_RESULTS_DIR / 'datasets' / 'splits_v1_2')
config['data']['num_workers'] = 2
config['training']['device'] = 'cuda'
config['training']['epochs'] = TARGET_EPOCHS
config['training']['batch_size'] = BATCH_SIZE
config['training']['mixed_precision'] = True
config['training']['resume'] = True
config['evaluation']['batch_size'] = BATCH_SIZE

config['paths']['output_dir'] = str(DRIVE_RESULTS_DIR)
config['paths']['checkpoint_dir'] = str(DRIVE_RESULTS_DIR / 'checkpoints')
config['paths']['sample_dir'] = str(DRIVE_RESULTS_DIR / 'samples')
config['paths']['plot_dir'] = str(DRIVE_RESULTS_DIR / 'plots')
config['paths']['evaluation_dir'] = str(DRIVE_RESULTS_DIR / 'evaluation')
config['paths']['log_dir'] = str(DRIVE_RESULTS_DIR / 'runs')

COLAB_CONFIG = DRIVE_RESULTS_DIR / 'config_colab_v1_2.yaml'
with COLAB_CONFIG.open('w', encoding='utf-8') as file:
    yaml.safe_dump(config, file, allow_unicode=True, sort_keys=False)

for directory in config['paths'].values():
    Path(directory).mkdir(parents=True, exist_ok=True)

print('Colab 配置：', COLAB_CONFIG)
print('目标 epoch：', config['training']['epochs'])
print('Batch Size：', config['training']['batch_size'])
print('EMA decay：', config['training']['ema_decay'])
print('pHash 阈值：', config['data']['phash_threshold'])
print('采样器：', config['sampling']['solver'], config['sampling']['num_steps'], '步')

## 7. 运行单元测试（建议首次执行）

这不会训练模型，也不会扫描全部数据。若测试失败，不要继续训练。

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=PROJECT_DIR,
    check=True,
)

## 8. 生成 V1.2 数据清单与 pHash 去重报告

该步骤不会删除 ZIP 或 `Data` 中的任何图片，只会在 Drive 结果目录中生成训练/验证/测试清单和 `dedup_report.json`。首次运行或调整 pHash 参数后执行；平时保持 `FORCE_REBUILD_SPLITS = False` 可复用缓存。

In [ ]:
split_command = [
    sys.executable, '-m', 'datasets.split_dataset',
    '--config', str(COLAB_CONFIG),
]
if FORCE_REBUILD_SPLITS:
    split_command.append('--force')
subprocess.run(split_command, cwd=PROJECT_DIR, check=True)

import json
split_dir = Path(config['data']['split_dir'])
metadata = json.loads((split_dir / 'metadata.json').read_text(encoding='utf-8'))
print('原始图片：', metadata['total_files'])
print('完全重复：', metadata['exact_duplicate_files'])
print('近似重复：', metadata['near_duplicate_files'])
print('最终保留：', metadata['unique_images'])
print('数据划分：', metadata['splits'])
print('去重报告：', split_dir / 'dedup_report.json')

## 9. 启动 TensorBoard（可选）

可以在训练前或训练结束后执行。日志直接读取 Drive 中的持久化目录。

In [ ]:
TENSORBOARD_LOG_DIR = str(DRIVE_RESULTS_DIR / 'runs')
%load_ext tensorboard
%tensorboard --logdir $TENSORBOARD_LOG_DIR --port 6006

## 10. 开始或继续训练

V1.2 将使用 Scale-Shift、EMA、余弦学习率和 Heun 预览。该单元格只在当前 Colab 会话中临时注入 Notebook 原生进度条，不修改本地训练行为：外层显示总 Epoch，内层显示当前训练/验证 batch，并实时展示 loss、学习率和预计剩余时间。每个 epoch 都会更新 Drive 中的 `latest.pt`；断线后重新执行第 1～6 节和本节即可恢复。首次训练不要把 V1.0/V1.1 checkpoint 复制到 V1.2 结果目录。

In [ ]:
import builtins
import os
import time
from IPython.display import Markdown, display
from tqdm.notebook import tqdm as notebook_tqdm

project_path = str(PROJECT_DIR)
if project_path not in sys.path:
    sys.path.insert(0, project_path)
os.chdir(PROJECT_DIR)

import train as train_module
from utils.config import load_config

# 只在当前 Notebook 单元格中替换 train 模块使用的 range/tqdm。
# finally 会恢复原对象，因此不会写回或改变本地 train.py 的行为。
original_tqdm = train_module.tqdm
original_train_one_epoch = train_module.train_one_epoch
original_evaluate_loss = train_module.evaluate_loss
progress_state = {'epoch_bar': None, 'train': None, 'val': None, 'lr': None}

def colab_epoch_range(*args):
    values = builtins.range(*args)
    start_epoch = values.start
    target_epoch = max(values.stop - 1, 0)
    epoch_bar = notebook_tqdm(
        values,
        desc='Epoch',
        unit='epoch',
        initial=max(start_epoch - 1, 0),
        total=target_epoch,
    )
    progress_state['epoch_bar'] = epoch_bar
    return epoch_bar

def colab_train_one_epoch(*args, **kwargs):
    train_loss, global_step = original_train_one_epoch(*args, **kwargs)
    progress_state['train'] = train_loss
    epoch_bar = progress_state['epoch_bar']
    optimizer = kwargs.get('optimizer')
    if epoch_bar is not None and optimizer is not None:
        progress_state['lr'] = optimizer.param_groups[0]['lr']
        epoch_bar.set_postfix(
            train=f'{train_loss:.4f}',
            lr=f"{optimizer.param_groups[0]['lr']:.2e}",
            refresh=True,
        )
    return train_loss, global_step

def colab_evaluate_loss(*args, **kwargs):
    val_loss = original_evaluate_loss(*args, **kwargs)
    progress_state['val'] = val_loss
    epoch_bar = progress_state['epoch_bar']
    if epoch_bar is not None:
        postfix = {'val': f'{val_loss:.4f}'}
        if progress_state['train'] is not None:
            postfix['train'] = f"{progress_state['train']:.4f}"
        if progress_state['lr'] is not None:
            postfix['lr'] = f"{progress_state['lr']:.2e}"
        epoch_bar.set_postfix(postfix, refresh=True)
    return val_loss

display(Markdown(
    f'**训练目标：{TARGET_EPOCHS} epochs · Batch Size：{BATCH_SIZE} · '
    f'结果目录：`{DRIVE_RESULTS_DIR}`**'
))
training_started = time.time()
try:
    train_module.range = colab_epoch_range
    train_module.tqdm = notebook_tqdm
    train_module.train_one_epoch = colab_train_one_epoch
    train_module.evaluate_loss = colab_evaluate_loss
    train_module.run_training(load_config(COLAB_CONFIG))
finally:
    if hasattr(train_module, 'range'):
        delattr(train_module, 'range')
    train_module.tqdm = original_tqdm
    train_module.train_one_epoch = original_train_one_epoch
    train_module.evaluate_loss = original_evaluate_loss
elapsed_minutes = (time.time() - training_started) / 60
display(Markdown(f'✅ **训练单元格执行完成，用时 {elapsed_minutes:.2f} 分钟。**'))

## 11. 查看 Drive 中的 checkpoint、loss 图和最新样本

In [ ]:
from IPython.display import display
from PIL import Image

checkpoint_dir = DRIVE_RESULTS_DIR / 'checkpoints'
for checkpoint_path in sorted(checkpoint_dir.glob('*.pt')):
    print(checkpoint_path.name, round(checkpoint_path.stat().st_size / 1024**2, 1), 'MiB')

loss_curve = DRIVE_RESULTS_DIR / 'plots' / 'loss_curve.png'
if loss_curve.is_file():
    display(Image.open(loss_curve))

sample_paths = sorted((DRIVE_RESULTS_DIR / 'samples').glob('epoch_*.png'))
if sample_paths:
    print('最新训练样本：', sample_paths[-1])
    display(Image.open(sample_paths[-1]))

## 12. 使用最佳 EMA 权重独立生成图片

`sample.py` 默认优先加载 checkpoint 中的 EMA 权重。修改 `SAMPLE_SEED` 可以生成不同图片。

In [ ]:
SAMPLE_SEED = 123
BEST_CHECKPOINT = DRIVE_RESULTS_DIR / 'checkpoints' / 'best.pt'
assert BEST_CHECKPOINT.is_file(), f'找不到最佳模型：{BEST_CHECKPOINT}'
INDEPENDENT_SAMPLE = DRIVE_RESULTS_DIR / 'samples' / f'ema_seed_{SAMPLE_SEED}.png'

subprocess.run(
    [
        sys.executable, 'sample.py',
        '--config', str(COLAB_CONFIG),
        '--checkpoint', str(BEST_CHECKPOINT),
        '--seed', str(SAMPLE_SEED),
        '--output', str(INDEPENDENT_SAMPLE),
    ],
    cwd=PROJECT_DIR,
    check=True,
)
display(Image.open(INDEPENDENT_SAMPLE))

## 13. 计算固定 flow MSE、FID 和 KID

评估默认使用 EMA 权重、固定验证随机数和固定生成噪声。FID/KID 会处理整个测试集，Heun 50 步生成所需时间较长；首次运行 torch-fidelity 时可能下载 Inception 权重。结果写入 Drive 的 `evaluation/metrics.json`。

In [ ]:
subprocess.run(
    [
        sys.executable, 'evaluate.py',
        '--config', str(COLAB_CONFIG),
        '--checkpoint', str(BEST_CHECKPOINT),
    ],
    cwd=PROJECT_DIR,
    check=True,
)

metrics_path = DRIVE_RESULTS_DIR / 'evaluation' / 'metrics.json'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print(json.dumps(metrics, ensure_ascii=False, indent=2))

## Colab 断线恢复与使用建议

- 新运行时需要重新执行第 1～6 节，将 ZIP 解压到新的 `/content`。
- Drive 结果目录不变时，训练会读取 `checkpoints/latest.pt` 自动恢复。
- `TARGET_EPOCHS` 是最终目标。例如 checkpoint 已到 epoch 40，设置为 100 会继续训练到 100。
- 如果 Batch Size 32 显存不足，可改成 16；模型结构和 checkpoint 仍兼容。
- 调整 `phash_threshold` 或 `phash_crop_ratios` 后，将 `FORCE_REBUILD_SPLITS` 改为 `True`，重新生成清单；确认完成后再改回 `False`。
- 不要删除 Drive 结果目录中的 `latest.pt`、`best.pt`、配置和数据清单。
- FID 在几百张测试图上波动较大，当前数据规模应优先结合 KID、固定 seed 图片和人工观察判断质量。